# FULL VERISET HAZIRLAMA
Tum verisetlerini birlestirelim


In [1]:
import pandas as pd
import numpy as np
import os
import re

print('Kutuphaneler yuklendi!')


Kutuphaneler yuklendi!


In [2]:
df_human_old = pd.read_csv('temizlenmis_turkce_veri.csv')
print(f'Mevcut veri: {len(df_human_old)} metin')
df_human_old.head()


Mevcut veri: 38786 metin


,content,category,headline
0,"Dışişleri Bakanı Davutoğlu, Yunanistan ile Tür...",dünya,'Ortak vizyonumuz var'
1,İsrail Gazze Şeridi nin kuzeyindeki bir tarlay...,dünya,İsrail'den Gazze Şeridi'ne hava saldırısı
2,Lübnan ın başkenti Beyrut ta düzenlenen bombal...,dünya,Cenaze için geniş güvenlik önlemleri alındı
3,KKTC de Sendikal Platform genel grev başlattı....,dünya,Gözaltındaki sendikacılar serbest
4,"Türkiye den yola çıkan Başak Bulut, Seçil Öznu...",dünya,Bisikletle Asya'da 3 bin kilometre yol katettiler


In [3]:
with open('/Users/yusufserdaroglu/Desktop/turkish-sentences-dataset-main/wiki.tr.txt', 'r', encoding='utf-8') as f:
    wiki_sentences = [line.strip() for line in f if line.strip()]

print(f'Wikipedia: {len(wiki_sentences)} cumle')


Wikipedia: 170458 cumle


In [4]:
def filter_text(text, min_words=5, max_words=30):
    if not isinstance(text, str):
        return False
    return min_words <= len(text.split()) <= max_words

df_human_old_filtered = df_human_old[df_human_old['content'].apply(filter_text)]
wiki_filtered = [s for s in wiki_sentences if filter_text(s)]

human_texts = df_human_old_filtered['content'].tolist() + wiki_filtered
human_texts = list(set(human_texts))

print(f'Toplam: {len(human_texts)}')


Toplam: 170596


In [5]:
def clean_text(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

ttc_path = 'TTC-3600_Orj'
categories = ['ekonomi', 'kultursanat', 'saglik', 'siyaset', 'spor', 'teknoloji']
ttc_texts = []

for category in categories:
    cat_path = os.path.join(ttc_path, category)
    files = [f for f in os.listdir(cat_path) if f.endswith('.txt')]
    print(f'{category}: {len(files)} dosya')
    for file_name in files:
        with open(os.path.join(cat_path, file_name), 'r', encoding='utf-8', errors='ignore') as f:
            text = clean_text(f.read())
            if len(text) >= 20:
                ttc_texts.append(text)

print(f'TTC-3600: {len(ttc_texts)} metin')


ekonomi: 600 dosya
kultursanat: 600 dosya
saglik: 600 dosya
siyaset: 600 dosya
spor: 600 dosya
teknoloji: 600 dosya
TTC-3600: 3600 metin


In [6]:
all_human_texts = human_texts + ttc_texts
all_human_texts = list(set(all_human_texts))

df_human_final = pd.DataFrame({'content': all_human_texts, 'label': 0})

print(f'TOPLAM INSAN: {len(df_human_final)}')
df_human_final.head()


TOPLAM INSAN: 174004


,content,label
0,Einstein Toplam kuralı kapalı toplamını bırakı...,0
1,Bazı ilaçlar böbrek fonksiyonlarını birden çok...,0
2,Plüton konusunda bilimsel anlamda bir bildiri ...,0
3,Miras hakkı için kardeşleri ile mücadele etti ...,0
4,"Burada ""c"" ışık hızı olduğundan bu koordinat y...",0


In [7]:
df_ai = pd.read_csv('ai_texts_35k.csv')
df_ai['label'] = 1

min_size = min(len(df_human_final), len(df_ai))
print(f'Her siniftan {min_size} ornek')

df_human_sampled = df_human_final.sample(n=min_size, random_state=42)
df_ai_sampled = df_ai.sample(n=min_size, random_state=42)

df_final = pd.concat([df_human_sampled, df_ai_sampled], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

output_file = 'final_dataset_SUPER.csv'
df_final[['content', 'label']].to_csv(output_file, index=False, encoding='utf-8-sig')

print(f'\nTAMAMLANDI!')
print(f'Toplam: {len(df_final)}')
print(f'Insan: {sum(df_final["label"] == 0)}')
print(f'AI: {sum(df_final["label"] == 1)}')
print(f'Kaydedildi: {output_file}')

df_final.head()


Her siniftan 35000 ornek

TAMAMLANDI!
Toplam: 70000
Insan: 35000
AI: 35000
Kaydedildi: final_dataset_SUPER.csv


,content,label
0,Uzmanlara göre sağlık alanında dijital dönüşüm...,1
1,Yakın dönemde psikoloji alanında teknoloji tem...,1
2,Yeni veriler ışığında lojistik alanında çevre ...,1
3,Aynı zamanda bu bölgenin başkentidir.,0
4,Stratejik açıdan bakıldığında iletişim alanınd...,1
